# Colab用ノートブック版

元の Python スクリプトをノートブック形式に変換したものです。  
Colab でそのまま上から順に実行できるようにしてあり、`Numba` のデコレータは notebook 互換のため `cache=False` に調整しています。


Phase 1: β条件付き拡散モデル用データセット生成
===============================================

╔══════════════════════════════════════════════════════════════════════╗
║  メタ認知的自己監査フレームワーク                                    ║
║                                                                      ║
║  各設計判断に対して3つの問いを立てる:                                ║
║    Q1. 何を仮定しているか（隠れた前提）                             ║
║    Q2. その仮定が崩れるとき何が起きるか（失敗モード）               ║
║    Q3. 崩れたことをどうやって検出するか（検証手段）                 ║
║                                                                      ║
║  これは TopoCycleGAN failure analysis の教訓の直接適用:             ║
║  「監査が監査対象と同じ歪みの中で行われれば、構造的な歪みを         ║
║    検出できない」→ 各判断に独立な外部アンカーを置く                 ║
╚══════════════════════════════════════════════════════════════════════╝

# Phase 1: データセット生成

β条件付き拡散モデル（DDPM）のための 32×32 Polyakov マップ生成。
Phase 0 で確認した β_c ≈ 1.8 を前提に、閉じ込め〜脱閉じ込めの
全領域をカバーするデータセットを構築する。

In [ ]:
# 1. 設計判断の監査（コード実行前に全て明示する）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

AUDIT = """
┌─────────────────────────────────────────────────────────────┐
│ 判断 1: データ表現 — (Re P, Im P) の2チャンネル             │
├─────────────────────────────────────────────────────────────┤
│ Q1: P(x,y) = cos θ + i sin θ（|P|=1 at each site）を仮定。│
│     → Re² + Im² = 1 というユニット円拘束が存在。           │
│     → 拡散モデルはこの拘束を自動的には学ばない。           │
│                                                             │
│ Q2: 拘束を破る生成 → 非物理的な P(x,y) が出る。           │
│     特に |Re²+Im²| ≠ 1 の点が生成される可能性。           │
│                                                             │
│ Q3: 生成後に各サイトの |P(x,y)|² を計算し、               │
│     1.0 からの偏差の分布をプロットする。                    │
│     → これが Phase 3 の物理的検証の第一歩になる。          │
│                                                             │
│ 代替案: 位相角 θ(x,y) ∈ [-π,π) を1チャンネルで保存。     │
│   利点: 拘束が自動的に満たされる（cos/sin で復元）。       │
│   欠点: ±π での不連続 → 拡散モデルが境界アーティファクト   │
│         を生む可能性。                                      │
│                                                             │
│ 判断: 両方保存する。θ を1チャンネル、(Re,Im) を2チャンネル │
│       Phase 2 でどちらが良いか実験的に判定。                │
├─────────────────────────────────────────────────────────────┤
│ 判断 2: 枚数 — 各β 500枚（raw）                            │
├─────────────────────────────────────────────────────────────┤
│ Q1: 「D4対称性で8倍増幅すれば4,000枚相当」を仮定。        │
│     → D4 は格子の厳密な対称性なので数学的に正しい。        │
│     → ただし augmentation は「新しい物理配置」ではない。    │
│                                                             │
│ Q2: 500枚では分布の裾が不十分 → 拡散モデルが                │
│     mode collapse を起こす可能性。                           │
│     特に β ≈ β_c 近傍では分布が広い（Phase 0 のヒストグラム│
│     参照：β=2.0 で |P| ∈ [0.15, 0.55]）。                 │
│                                                             │
│ Q3: 段階的スケーリング。まず 500枚で学習し、               │
│     FID（もし参照分布があれば）or 生成⟨|P|⟩の MC との      │
│     一致度で判定。不足なら 1,000 → 2,000 と増やす。        │
│     pass/fail 基準を事前に定義（下記）。                    │
├─────────────────────────────────────────────────────────────┤
│ 判断 3: β グリッド — 10点、β_c近傍を密に                    │
├─────────────────────────────────────────────────────────────┤
│ Q1: 拡散モデルが β の補間を学習できることを仮定。           │
│     → β が連続スカラー入力なら、学習点間の補間は合理的。   │
│     → ただし BKT 転移では β_c 近傍で分布が急変する。       │
│                                                             │
│ Q2: β_c 近傍の補間が崩れる → 転移点を跨ぐ生成が不正確。   │
│                                                             │
│ Q3: Phase 2 で β=1.75, 1.85（学習に含まない点）での         │
│     生成品質を検証。これが「内挿テスト」。                  │
│     → 失敗すれば β_c 近傍のグリッドを密にする。            │
├─────────────────────────────────────────────────────────────┤
│ 判断 4: 非相関性 — N_skip の β 依存性                       │
├─────────────────────────────────────────────────────────────┤
│ Q1: Phase 0 の τ_int をそのまま使えることを仮定。           │
│     → Phase 0 は N_meas=200。Phase 1 は N_meas=500 で       │
│       測定が長いので τ_int の推定精度は上がるはず。         │
│                                                             │
│ Q2: 実際の τ_int が Phase 0 より大きい                      │
│     → サンプル間に相関が残る → 実効サンプル数が期待より少  │
│     → 拡散モデルが配置の diversity を過大評価。             │
│                                                             │
│ Q3: 生成時に τ_int をリアルタイム推定し、N_skip を          │
│     max(50, 4×τ_int) に自動調整。結果の τ_int を保存。     │
├─────────────────────────────────────────────────────────────┤
│ 判断 5: pass/fail 基準（事前定義、事後変更禁止）            │
├─────────────────────────────────────────────────────────────┤
│ PASS 条件（全て満たす）:                                    │
│   ① 各βの ⟨|P|⟩ が Phase 0 の値と 2σ 以内で一致           │
│   ② プラケット ⟨cos P⟩ が Phase 0 と 1% 以内で一致        │
│   ③ 受理率が全βで 45-75% の範囲内                          │
│   ④ β > 2.0 で ⟨|P|⟩ が単調増加                            │
│   ⑤ Polyakov マップの目視検査（閉じ込め/脱閉じ込め）       │
│                                                             │
│ これは Phase 0 の「外部アンカー」として機能する。           │
└─────────────────────────────────────────────────────────────┘
"""

In [ ]:
# 2. セットアップ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import numpy as np
import matplotlib.pyplot as plt
import time
import json
import os
from numba import njit

try:
    import google.colab
    IN_COLAB = True
    print("✓ Google Colab 環境を検出")
except ImportError:
    IN_COLAB = False
    print("ローカル環境で実行中")

# 出力ディレクトリ
OUTPUT_DIR = "phase1_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NumPy: {np.__version__}")
print(f"出力先: {OUTPUT_DIR}/")

✓ Google Colab 環境を検出
NumPy: 2.0.2
出力先: phase1_dataset/


In [ ]:
# 3. Numba コア関数（Phase 0 からそのまま移植）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@njit(cache=False)
def metropolis_sweep(links, beta, Ns, Nt, delta):
    accepted = 0
    total = 3 * Ns * Ns * Nt

    for mu in range(3):
        for x in range(Ns):
            for y in range(Ns):
                for t in range(Nt):
                    old_theta = links[mu, x, y, t]
                    new_theta = old_theta + delta * (2.0 * np.random.random() - 1.0)
                    dS = 0.0

                    for nu in range(3):
                        if nu == mu:
                            continue
                        dmu_x = 1 if mu == 0 else 0
                        dmu_y = 1 if mu == 1 else 0
                        dmu_t = 1 if mu == 2 else 0
                        dnu_x = 1 if nu == 0 else 0
                        dnu_y = 1 if nu == 1 else 0
                        dnu_t = 1 if nu == 2 else 0

                        xf = (x + dmu_x) % Ns; yf = (y + dmu_y) % Ns; tf = (t + dmu_t) % Nt
                        th_nu_fwd = links[nu, xf, yf, tf]
                        xn = (x + dnu_x) % Ns; yn = (y + dnu_y) % Ns; tn = (t + dnu_t) % Nt
                        th_mu_fwd = links[mu, xn, yn, tn]
                        th_nu_here = links[nu, x, y, t]
                        staple_fwd = th_nu_fwd - th_mu_fwd - th_nu_here
                        dS += -beta * (np.cos(new_theta + staple_fwd)
                                     - np.cos(old_theta + staple_fwd))

                        xb = (x - dnu_x) % Ns; yb = (y - dnu_y) % Ns; tb = (t - dnu_t) % Nt
                        th_mu_bwd = links[mu, xb, yb, tb]
                        xbf = (x - dnu_x + dmu_x) % Ns
                        ybf = (y - dnu_y + dmu_y) % Ns
                        tbf = (t - dnu_t + dmu_t) % Nt
                        th_nu_bwd = links[nu, xbf, ybf, tbf]
                        th_nu_at_b = links[nu, xb, yb, tb]
                        staple_bwd = th_mu_bwd + th_nu_bwd - th_nu_at_b
                        dS += -beta * (np.cos(staple_bwd - new_theta)
                                     - np.cos(staple_bwd - old_theta))

                    if dS <= 0.0 or np.random.random() < np.exp(-dS):
                        links[mu, x, y, t] = new_theta
                        accepted += 1
    return accepted / total


@njit(cache=False)
def wrap_phases(links):
    for i in range(links.shape[0]):
        for j in range(links.shape[1]):
            for k in range(links.shape[2]):
                for l in range(links.shape[3]):
                    links[i, j, k, l] = (links[i, j, k, l] + np.pi) % (2 * np.pi) - np.pi
    return links


@njit(cache=False)
def measure_polyakov(links, Ns, Nt):
    poly_re = np.zeros((Ns, Ns))
    poly_im = np.zeros((Ns, Ns))
    for x in range(Ns):
        for y in range(Ns):
            phase_sum = 0.0
            for t in range(Nt):
                phase_sum += links[2, x, y, t]
            poly_re[x, y] = np.cos(phase_sum)
            poly_im[x, y] = np.sin(phase_sum)
    avg_re = np.sum(poly_re) / (Ns * Ns)
    avg_im = np.sum(poly_im) / (Ns * Ns)
    abs_poly = np.sqrt(avg_re * avg_re + avg_im * avg_im)
    return poly_re, poly_im, avg_re, avg_im, abs_poly


@njit(cache=False)
def measure_plaquette(links, Ns, Nt):
    plaq_sum = 0.0
    count = 0
    for mu in range(3):
        for nu in range(mu + 1, 3):
            dmu_x = 1 if mu == 0 else 0
            dmu_y = 1 if mu == 1 else 0
            dmu_t = 1 if mu == 2 else 0
            dnu_x = 1 if nu == 0 else 0
            dnu_y = 1 if nu == 1 else 0
            dnu_t = 1 if nu == 2 else 0
            for x in range(Ns):
                for y in range(Ns):
                    for t in range(Nt):
                        xf = (x + dmu_x) % Ns; yf = (y + dmu_y) % Ns; tf = (t + dmu_t) % Nt
                        xn = (x + dnu_x) % Ns; yn = (y + dnu_y) % Ns; tn = (t + dnu_t) % Nt
                        P = (links[mu, x, y, t] + links[nu, xf, yf, tf]
                           - links[mu, xn, yn, tn] - links[nu, x, y, t])
                        plaq_sum += np.cos(P)
                        count += 1
    return plaq_sum / count


@njit(cache=False)
def measure_polyakov_phase(links, Ns, Nt):
    """位相角 θ(x,y) = Σ_t links[2,x,y,t] を返す。[-π, π) に wrap。"""
    phase_map = np.zeros((Ns, Ns))
    for x in range(Ns):
        for y in range(Ns):
            s = 0.0
            for t in range(Nt):
                s += links[2, x, y, t]
            phase_map[x, y] = (s + np.pi) % (2 * np.pi) - np.pi
    return phase_map


print("コア関数を定義しました。")

コア関数を定義しました。


In [ ]:
# 4. Phase 0 から移植した補助関数 + adaptive delta
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def tune_delta_v2(delta, acc_rate, target=(0.50, 0.70),
                  shrink=0.90, grow=1.10,
                  emergency_shrink=0.70, emergency_grow=1.30,
                  delta_min=0.02, delta_max=np.pi):
    low, high = target
    new_delta = delta
    if acc_rate < 0.20:
        new_delta *= emergency_shrink
    elif acc_rate < low:
        new_delta *= shrink
    elif acc_rate > 0.85:
        new_delta *= emergency_grow
    elif acc_rate > high:
        new_delta *= grow
    return min(max(new_delta, delta_min), delta_max)


def physics_based_delta0(beta):
    return min(np.pi, 2.0 / np.sqrt(max(beta, 0.1)))


def autocorr_1d(x, max_lag=None):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 2:
        return np.array([1.0])
    x = x - np.mean(x)
    var = np.dot(x, x) / n
    if var <= 0:
        return np.ones(1)
    if max_lag is None:
        max_lag = min(n // 2, 200)
    acf = np.empty(max_lag + 1, dtype=np.float64)
    acf[0] = 1.0
    for lag in range(1, max_lag + 1):
        acf[lag] = np.dot(x[:-lag], x[lag:]) / ((n - lag) * var)
    return acf


def integrated_autocorr_time(x, c=5.0, max_lag=None):
    acf = autocorr_1d(x, max_lag=max_lag)
    tau = 0.5
    for t in range(1, len(acf)):
        if acf[t] <= 0:
            break
        tau += acf[t]
        if t > c * tau:
            break
    return max(tau, 0.5), acf

In [ ]:
# 5. パラメータ設定
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 格子
Ns = 32
Nt = 4

# β グリッド設計
# ┌──────────────────────────────────────────────────────────┐
# │ 監査: β_c ≈ 1.8 の根拠は Phase 0 の d⟨|P|⟩/dβ ピーク。 │
# │ グリッドは β_c の ±0.3 を密に（Δβ=0.1-0.2）、          │
# │ 遠方を疎に（Δβ=1.0-4.0）配置。                         │
# │                                                          │
# │ 文献チェック: Ns=32, Nt=4 の 2+1D compact U(1) での      │
# │ β_c の先行研究値は β_c ≈ 1.0113 (infinite volume) だが、│
# │ これは Ns,Nt→∞ の外挿値。Nt=4 有限体積では β_c が      │
# │ 上方シフトするのは整合的。                               │
# └──────────────────────────────────────────────────────────┘

BETA_GRID = {
    # β: (N_meas, N_skip)  — β_c 近傍は N_skip を増やす
    0.5:  (500,  50),   # 深い閉じ込め（τ_int ≈ 0.5）
    1.0:  (500,  50),   # 閉じ込め
    1.5:  (500,  50),   # 閉じ込め〜転移手前（τ_int ≈ 0.6）
    1.7:  (500,  80),   # 転移直前（τ_int ≈ 1.5）
    1.9:  (500, 100),   # 転移直後（τ_int ≈ 5.3 → 4τ=21、N_skip=100で十分）
    2.0:  (500, 100),   # 臨界近傍（τ_int ≈ 14 → 4τ=56、N_skip=100で ≈ 2独立）
    2.5:  (500,  50),   # 脱閉じ込め（τ_int ≈ 1.4）
    4.0:  (500,  50),   # 脱閉じ込め（τ_int ≈ 1.3）
    6.0:  (500,  50),   # 深い脱閉じ込め
   10.0:  (500,  50),   # 最深脱閉じ込め
}

# ┌──────────────────────────────────────────────────────────┐
# │ 監査: β=2.0 で N_skip=100 は 4τ_int ≈ 56 に対して      │
# │ まだ不足（100/56 ≈ 1.8 独立/サンプル）。               │
# │ → 500枚のうち実効独立サンプルは ≈ 500 × 1.8/2 ≈ 450。 │
# │ → D4 で 3,600。不足なら N_skip=150 に上げる。          │
# │ → ただし β=2.0 は転移点なので拡散モデルにとって        │
# │   最も難しい点。ここの品質は Phase 2 で個別検証する。   │
# └──────────────────────────────────────────────────────────┘

# 熱平衡化パラメータ
N_TUNE = 2000
N_THERM_FIXED = 2000

# Phase 0 参照値（pass/fail 検証用）
PHASE0_REFERENCE = {
    # β: (⟨|P|⟩, err, ⟨plaq⟩)
    1.5:  (0.0613, 0.0023, 0.6880),
    1.7:  (0.1091, 0.0053, 0.7502),
    1.9:  (0.3133, 0.0195, 0.7931),
    2.0:  (0.3789, 0.0254, 0.8077),
    2.5:  (0.5657, 0.0040, 0.8543),
    4.0:  (0.7159, 0.0037, 0.9127),
    6.0:  (0.8115, 0.0025, 0.9428),
   10.0:  (0.8858, 0.0015, 0.9661),
}

beta_list_sorted = sorted(BETA_GRID.keys())
total_samples = sum(v[0] for v in BETA_GRID.values())
total_sweeps = sum(v[0] * v[1] for v in BETA_GRID.values())

print(f"格子: {Ns}×{Ns}×{Nt}")
print(f"β points: {len(BETA_GRID)}")
print(f"β grid: {beta_list_sorted}")
print(f"Total raw samples: {total_samples}")
print(f"After D4 augmentation: {total_samples * 8}")
print(f"Total sweeps (measurement only): {total_sweeps:,}")

格子: 32×32×4
β points: 10
β grid: [0.5, 1.0, 1.5, 1.7, 1.9, 2.0, 2.5, 4.0, 6.0, 10.0]
Total raw samples: 5000
After D4 augmentation: 40000
Total sweeps (measurement only): 315,000


In [ ]:
# 6. JIT ウォームアップ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("Numba JIT コンパイル中 ...")
t0 = time.time()
_dummy = np.random.uniform(-np.pi, np.pi, size=(3, 4, 4, 2))
_ = metropolis_sweep(_dummy, 1.0, 4, 2, 1.0)
_ = wrap_phases(_dummy)
_ = measure_polyakov(_dummy, 4, 2)
_ = measure_plaquette(_dummy, 4, 2)
_ = measure_polyakov_phase(_dummy, 4, 2)
print(f"JIT 完了: {time.time() - t0:.1f} 秒")

Numba JIT コンパイル中 ...
JIT 完了: 4.1 秒


In [ ]:
# 7. データ生成メインループ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_dataset(beta_grid, Ns, Nt, output_dir,
                     n_tune=2000, n_therm_fixed=2000, seed=42):
    """
    全βのデータを生成し、β毎に .npz ファイルを保存。

    保存形式:
      {output_dir}/beta_{beta:.4f}.npz
        - poly_re:    (N_meas, Ns, Ns) float32 — Re P(x,y)
        - poly_im:    (N_meas, Ns, Ns) float32 — Im P(x,y)
        - poly_phase: (N_meas, Ns, Ns) float32 — θ(x,y) ∈ [-π,π)
        - poly_abs:   (N_meas,) float32 — |⟨P⟩| per config
        - plaquette:  (N_meas,) float32 — ⟨cos P⟩ per config
        - metadata:   dict (β, delta, acc, τ_int, etc.)

    ┌──────────────────────────────────────────────────────┐
    │ 監査: なぜ3種類の表現を全て保存するか               │
    │ → Phase 2 でどの表現が拡散モデルに最適かは未知。    │
    │   事前に決め打ちするのではなく、実験で判定する。     │
    │   ストレージコスト: 500 × 32×32 × 3 × 4B ≈ 6MB/β。 │
    │   10β で 60MB。無視できる。                          │
    └──────────────────────────────────────────────────────┘
    """
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    all_results = {}
    total_start = time.time()

    links = None
    prev_delta_final = None

    for ib, beta in enumerate(sorted(beta_grid.keys())):
        n_meas, n_skip = beta_grid[beta]

        print(f"\n{'='*72}")
        print(f"[Phase 1] {ib+1}/{len(beta_grid)} : β = {beta:.2f}")
        print(f"  N_meas={n_meas}, N_skip={n_skip}")
        print(f"{'='*72}")

        # --- 初期化 ---
        if ib == 0:
            links = rng.uniform(-np.pi, np.pi, size=(3, Ns, Ns, Nt))
            print("  初期化: hot start")
        else:
            print("  初期化: 前回βの最終配置を継承")

        # --- delta ---
        if prev_delta_final is not None:
            delta0 = prev_delta_final
            print(f"  delta0: {delta0:.4f} (継承)")
        else:
            delta0 = physics_based_delta0(beta)
            print(f"  delta0: {delta0:.4f} (2/√β)")

        # --- 熱平衡化 ---
        t0 = time.time()
        delta = float(delta0)

        # Tuning phase
        for i in range(n_tune):
            acc = metropolis_sweep(links, beta, Ns, Nt, delta)
            if (i + 1) % 20 == 0:
                links = wrap_phases(links)
                delta = tune_delta_v2(delta, acc)

        # Fixed-delta thermalization
        for i in range(n_therm_fixed):
            acc = metropolis_sweep(links, beta, Ns, Nt, delta)
            if (i + 1) % 100 == 0:
                links = wrap_phases(links)

        links = wrap_phases(links)
        t_therm = time.time() - t0
        print(f"  熱平衡化完了: {t_therm:.0f}秒, delta={delta:.4f}")

        # --- 測定 ---
        t0 = time.time()

        poly_re_all = np.empty((n_meas, Ns, Ns), dtype=np.float32)
        poly_im_all = np.empty((n_meas, Ns, Ns), dtype=np.float32)
        poly_phase_all = np.empty((n_meas, Ns, Ns), dtype=np.float32)
        poly_abs_all = np.empty(n_meas, dtype=np.float32)
        plaq_all = np.empty(n_meas, dtype=np.float32)
        acc_all = np.empty(n_meas, dtype=np.float32)

        for m in range(n_meas):
            # Decorrelation sweeps
            local_acc = []
            for _ in range(n_skip):
                a = metropolis_sweep(links, beta, Ns, Nt, delta)
                local_acc.append(a)

            if (m + 1) % 20 == 0:
                links = wrap_phases(links)

            # Measurement
            poly_re, poly_im, avg_re, avg_im, abs_poly = measure_polyakov(links, Ns, Nt)
            phase_map = measure_polyakov_phase(links, Ns, Nt)
            plaq = measure_plaquette(links, Ns, Nt)

            poly_re_all[m] = poly_re.astype(np.float32)
            poly_im_all[m] = poly_im.astype(np.float32)
            poly_phase_all[m] = phase_map.astype(np.float32)
            poly_abs_all[m] = abs_poly
            plaq_all[m] = plaq
            acc_all[m] = np.mean(local_acc)

            if (m + 1) % 100 == 0:
                print(f"    {m+1}/{n_meas}: ⟨|P|⟩={np.mean(poly_abs_all[:m+1]):.4f}, "
                      f"acc={np.mean(acc_all[:m+1]):.3f}")

        t_meas = time.time() - t0
        prev_delta_final = delta

        # --- τ_int の実測 ---
        tau_int, _ = integrated_autocorr_time(poly_abs_all.astype(np.float64))

        # ┌──────────────────────────────────────────────────┐
        # │ 監査チェック: τ_int vs N_skip                    │
        # │ N_skip が 2τ_int 未満なら警告を出す              │
        # └──────────────────────────────────────────────────┘
        if n_skip < 2 * tau_int:
            print(f"  ⚠ WARNING: N_skip={n_skip} < 2τ_int={2*tau_int:.0f}")
            print(f"    → サンプル間に有意な相関が残っている可能性")

        # --- 統計量 ---
        poly_mean = float(np.mean(poly_abs_all))
        poly_err = float(np.std(poly_abs_all) / np.sqrt(n_meas / max(1, 2*tau_int)))
        plaq_mean = float(np.mean(plaq_all))
        acc_mean = float(np.mean(acc_all))

        # --- 保存 ---
        metadata = {
            "beta": float(beta),
            "Ns": Ns, "Nt": Nt,
            "n_meas": n_meas, "n_skip": n_skip,
            "delta_final": float(delta),
            "poly_mean": poly_mean,
            "poly_err": poly_err,
            "plaquette_mean": plaq_mean,
            "tau_int": float(tau_int),
            "acc_mean": acc_mean,
            "time_therm_sec": float(t_therm),
            "time_meas_sec": float(t_meas),
            "seed": seed,
        }

        fname = os.path.join(output_dir, f"beta_{beta:.4f}.npz")
        np.savez_compressed(fname,
            poly_re=poly_re_all,
            poly_im=poly_im_all,
            poly_phase=poly_phase_all,
            poly_abs=poly_abs_all,
            plaquette=plaq_all,
        )

        all_results[float(beta)] = metadata

        fsize_mb = os.path.getsize(fname) / 1024**2
        print(f"  -> ⟨|P|⟩ = {poly_mean:.4f} ± {poly_err:.4f}")
        print(f"  -> ⟨plaq⟩ = {plaq_mean:.5f}")
        print(f"  -> τ_int = {tau_int:.1f}, acc = {acc_mean:.3f}")
        print(f"  -> 保存: {fname} ({fsize_mb:.1f} MB)")
        print(f"  -> 測定時間: {t_meas:.0f}秒")

    # --- メタデータ JSON ---
    meta_path = os.path.join(output_dir, "dataset_metadata.json")
    with open(meta_path, "w") as f:
        json.dump({
            "params": {"Ns": Ns, "Nt": Nt, "seed": seed},
            "beta_results": {str(k): v for k, v in all_results.items()},
            "total_time_min": (time.time() - total_start) / 60,
        }, f, indent=2)

    elapsed = time.time() - total_start
    print(f"\n{'='*72}")
    print(f"Phase 1 データ生成完了: {elapsed/60:.1f} 分")
    print(f"メタデータ: {meta_path}")
    print(f"{'='*72}")

    return all_results


# 実行
results = generate_dataset(BETA_GRID, Ns, Nt, OUTPUT_DIR, seed=42)


[Phase 1] 1/10 : β = 0.50
  N_meas=500, N_skip=50
  初期化: hot start
  delta0: 2.8284 (2/√β)
  熱平衡化完了: 26秒, delta=2.8284
    100/500: ⟨|P|⟩=0.0286, acc=0.677
    200/500: ⟨|P|⟩=0.0277, acc=0.677


KeyboardInterrupt: 

In [ ]:
# 8. pass/fail 検証（Phase 0 との整合性チェック）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def verify_against_phase0(results, reference):
    """
    ┌──────────────────────────────────────────────────────┐
    │ 監査: この関数は「事前に定義した基準」のみで判定。   │
    │ 結果を見てから基準を変えてはいけない。               │
    │ （failure analysis §5.3 の教訓）                     │
    └──────────────────────────────────────────────────────┘
    """
    print("\n" + "="*72)
    print("Phase 0 整合性検証")
    print("="*72)

    all_pass = True

    # ① ⟨|P|⟩ の一致（2σ以内）
    print("\n--- ① ⟨|P|⟩ 一致チェック (2σ) ---")
    for beta in sorted(reference.keys()):
        if beta not in results:
            continue
        ref_mean, ref_err, _ = reference[beta]
        new_mean = results[beta]["poly_mean"]
        new_err = results[beta]["poly_err"]

        # 両方の誤差を合成
        combined_err = np.sqrt(ref_err**2 + new_err**2)
        diff = abs(new_mean - ref_mean)
        n_sigma = diff / combined_err if combined_err > 0 else 0

        status = "✓" if n_sigma < 2.0 else "✗ FAIL"
        if n_sigma >= 2.0:
            all_pass = False
        print(f"  β={beta:.1f}: Phase0={ref_mean:.4f}±{ref_err:.4f}, "
              f"Phase1={new_mean:.4f}±{new_err:.4f}, "
              f"Δ={diff:.4f} ({n_sigma:.1f}σ) {status}")

    # ② plaquette の一致（1%以内）
    print("\n--- ② plaquette 一致チェック (1%) ---")
    for beta in sorted(reference.keys()):
        if beta not in results:
            continue
        _, _, ref_plaq = reference[beta]
        new_plaq = results[beta]["plaquette_mean"]
        pct_diff = abs(new_plaq - ref_plaq) / ref_plaq * 100
        status = "✓" if pct_diff < 1.0 else "✗ FAIL"
        if pct_diff >= 1.0:
            all_pass = False
        print(f"  β={beta:.1f}: Phase0={ref_plaq:.5f}, Phase1={new_plaq:.5f}, "
              f"差={pct_diff:.2f}% {status}")

    # ③ 受理率（45-75%）
    print("\n--- ③ 受理率チェック (45-75%) ---")
    for beta in sorted(results.keys()):
        acc = results[beta]["acc_mean"]
        status = "✓" if 0.45 <= acc <= 0.75 else "✗ FAIL"
        if not (0.45 <= acc <= 0.75):
            all_pass = False
        print(f"  β={beta:.2f}: acc={acc:.3f} {status}")

    # ④ 単調性（β > 2.0）
    print("\n--- ④ ⟨|P|⟩ 単調性 (β > 2.0) ---")
    high_betas = sorted([b for b in results.keys() if b > 2.0])
    monotone = True
    for i in range(1, len(high_betas)):
        if results[high_betas[i]]["poly_mean"] < results[high_betas[i-1]]["poly_mean"]:
            monotone = False
            print(f"  ✗ β={high_betas[i-1]:.1f}→{high_betas[i]:.1f}: "
                  f"{results[high_betas[i-1]]['poly_mean']:.4f} > {results[high_betas[i]]['poly_mean']:.4f}")
    if monotone:
        print("  ✓ 完全に単調")
    else:
        all_pass = False

    # 総合判定
    print(f"\n{'='*72}")
    if all_pass:
        print("★ PASS: Phase 1 データは Phase 0 と整合。拡散モデル学習に進めます。")
    else:
        print("✗ FAIL: Phase 0 との不整合あり。原因を特定してからデータを使用してください。")
    print(f"{'='*72}")

    return all_pass


passed = verify_against_phase0(results, PHASE0_REFERENCE)

In [ ]:
# 9. サンプル可視化
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def visualize_samples(output_dir, beta_grid):
    """各βから1枚ずつ Re P(x,y) を表示。"""
    betas = sorted(beta_grid.keys())
    n = len(betas)
    ncols = min(5, n)
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axes = np.array(axes).flatten()

    for i, beta in enumerate(betas):
        fname = os.path.join(output_dir, f"beta_{beta:.4f}.npz")
        data = np.load(fname)
        # ランダムなサンプルを1枚表示
        idx = np.random.randint(data["poly_re"].shape[0])
        im = axes[i].imshow(data["poly_re"][idx], cmap='RdBu_r',
                           vmin=-1, vmax=1, origin='lower',
                           interpolation='nearest')
        axes[i].set_title(f'β={beta:.2f}\n⟨|P|⟩={np.mean(data["poly_abs"]):.3f}',
                         fontsize=10)
        axes[i].set_xticks([]); axes[i].set_yticks([])

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Phase 1: Re P(x,y) samples", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "sample_gallery.png"),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("サンプルギャラリー保存完了")


visualize_samples(OUTPUT_DIR, BETA_GRID)

In [ ]:
# 10. D4 対称性増幅ユーティリティ（Phase 2 で使う）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def augment_d4(image):
    """
    32×32 画像に D4 変換（4回転 × 2反転 = 8枚）を適用。

    ┌──────────────────────────────────────────────────────┐
    │ 監査: D4 変換は格子の厳密な対称性か？                │
    │ → Ns×Ns×Nt 格子で Ns=Nt でないので、空間回転は     │
    │   (x,y) 平面内の回転のみ（時間軸は回転しない）。    │
    │   → D4 は空間 (x,y) の対称性 → ✓ 厳密。            │
    │   Polyakov ループは時間方向の積なので、               │
    │   空間回転に対して共変 → ✓ 増幅は正当。             │
    └──────────────────────────────────────────────────────┘
    """
    out = [image]
    for k in range(1, 4):
        out.append(np.rot90(image, k))
    flipped = np.fliplr(image)
    out.append(flipped)
    for k in range(1, 4):
        out.append(np.rot90(flipped, k))
    return out


# 検証: 8枚生成されるか
_test = np.random.randn(32, 32)
assert len(augment_d4(_test)) == 8, "D4 must produce 8 images"
print("D4 増幅ユーティリティ検証OK")

In [ ]:
# 11. PyTorch Dataset クラス（Phase 2 用、ここでは定義のみ）
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PYTORCH_DATASET_CODE = '''
import torch
from torch.utils.data import Dataset
import numpy as np
import os

class PolyakovDataset(Dataset):
    """
    Phase 1 で生成した .npz ファイルからの PyTorch Dataset。

    引数:
        data_dir: .npz ファイルが入ったディレクトリ
        channels: "re" (1ch), "re_im" (2ch), or "phase" (1ch)
        augment_d4: D4 対称性で 8 倍増幅するか
        normalize: [-1,1] → [0,1] に変換するか

    返り値:
        image: (C, 32, 32) tensor
        beta: () scalar tensor（条件付け用）
    """
    def __init__(self, data_dir, channels="re_im", augment_d4=True, normalize=False):
        self.files = sorted([
            os.path.join(data_dir, f) for f in os.listdir(data_dir)
            if f.endswith('.npz')
        ])
        self.channels = channels
        self.augment = augment_d4
        self.normalize = normalize

        # 全データをメモリにロード（60MB なので問題なし）
        self.images = []
        self.betas = []

        for fpath in self.files:
            data = np.load(fpath)
            # メタデータから β を取得
            beta = float(fpath.split('beta_')[1].split('.npz')[0])
            n = data['poly_re'].shape[0]

            for i in range(n):
                if channels == "re":
                    img = data['poly_re'][i:i+1]       # (1, 32, 32)
                elif channels == "re_im":
                    img = np.stack([data['poly_re'][i],
                                   data['poly_im'][i]])  # (2, 32, 32)
                elif channels == "phase":
                    img = data['poly_phase'][i:i+1]     # (1, 32, 32)

                if self.augment:
                    # D4 on spatial dims (last two)
                    for k in range(4):
                        rotated = np.rot90(img, k, axes=(-2, -1)).copy()
                        self.images.append(rotated)
                        self.betas.append(beta)
                    flipped = np.flip(img, axis=-1).copy()
                    for k in range(4):
                        rotated = np.rot90(flipped, k, axes=(-2, -1)).copy()
                        self.images.append(rotated)
                        self.betas.append(beta)
                else:
                    self.images.append(img.copy())
                    self.betas.append(beta)

        self.images = np.array(self.images, dtype=np.float32)
        self.betas = np.array(self.betas, dtype=np.float32)

        # β を [0, 1] に正規化（条件付けの安定性のため）
        self.beta_min = self.betas.min()
        self.beta_max = self.betas.max()
        self.betas_normed = (self.betas - self.beta_min) / (self.beta_max - self.beta_min)

        print(f"Dataset: {len(self)} samples, channels={channels}, "
              f"augment={augment_d4}")
        print(f"  β range: [{self.beta_min:.2f}, {self.beta_max:.2f}]")
        print(f"  Image shape: {self.images[0].shape}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx])
        beta = torch.tensor(self.betas_normed[idx])

        if self.normalize:
            img = (img + 1.0) / 2.0  # [-1,1] → [0,1]

        return img, beta

    def beta_to_normed(self, beta_physical):
        """物理的 β → 正規化 β（生成時に使用）"""
        return (beta_physical - self.beta_min) / (self.beta_max - self.beta_min)

    def normed_to_beta(self, beta_normed):
        """正規化 β → 物理的 β"""
        return beta_normed * (self.beta_max - self.beta_min) + self.beta_min
'''

print("PyTorch Dataset クラス定義（Phase 2 で使用）:")
print(f"  channels='re_im' → 2ch × D4 → {total_samples * 8} 枚")
print(f"  channels='phase'  → 1ch × D4 → {total_samples * 8} 枚")

In [ ]:
# 12. データセットサマリー出力
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def print_summary(results, output_dir):
    print("\n" + "="*72)
    print("Phase 1 データセットサマリー")
    print("="*72)

    total_files = 0
    total_size_mb = 0

    for beta in sorted(results.keys()):
        r = results[beta]
        fname = os.path.join(output_dir, f"beta_{beta:.4f}.npz")
        fsize = os.path.getsize(fname) / 1024**2 if os.path.exists(fname) else 0
        total_files += 1
        total_size_mb += fsize
        print(f"  β={beta:5.2f} | {r['n_meas']:4d} samples | "
              f"⟨|P|⟩={r['poly_mean']:.4f} | τ={r['tau_int']:5.1f} | "
              f"{fsize:.1f}MB")

    print(f"\n  合計: {total_files} ファイル, {total_size_mb:.1f} MB")
    print(f"  Raw samples: {sum(r['n_meas'] for r in results.values())}")
    print(f"  After D4: {sum(r['n_meas'] for r in results.values()) * 8}")

    print(f"\n  Phase 2 での使用方法:")
    print(f"    dataset = PolyakovDataset('{output_dir}', channels='re_im')")
    print(f"    loader = DataLoader(dataset, batch_size=64, shuffle=True)")
    print("="*72)


print_summary(results, OUTPUT_DIR)